# Exploring OpenAI Whisper in Mase

In this notebook, we import the [OpenAI Whisper](https://arxiv.org/abs/2212.04356) speech recognition model into Mase and inspect the FX graph that is generated. Whisper is an encoder-decoder Transformer model trained on a large dataset of diverse audio. We'll use `openai/whisper-tiny` to keep things lightweight.

We follow the same workflow as Tutorial 1:
1. Load a pretrained Whisper model from HuggingFace
2. Trace it into a `MaseGraph` using Torch FX
3. Visualize the compute graph
4. Raise the graph to the Mase IR with metadata passes
5. Run simple analysis and transform passes

## 1. Load the Whisper Encoder

`WhisperForConditionalGeneration` is not supported by HuggingFace's `symbolic_trace`, and the decoder contains dynamic ops (`torch.arange` with proxy device) that also break FX tracing. We therefore focus on the **encoder** — the most architecturally interesting part of Whisper, which converts mel spectrograms into hidden states via convolutions + a transformer.

We wrap the encoder in a plain `nn.Module` so MaseGraph uses its own `MaseTracer` (following the same pattern as the [Wav2Vec wrapper](https://github.com/DeepWok/mase/blob/main/src/chop/models/wav2vec/modelling_wave2vec.py) in Mase). We also mark `WhisperEncoderLayer` as a leaf module since the attention sublayers contain control flow (`if attn_output.size() != ...`) that can't be symbolically traced.

In [1]:
import torch
import torch.nn as nn
from transformers import WhisperModel, WhisperProcessor


class WhisperEncoderWrapper(nn.Module):
    """
    Wraps the Whisper encoder in a plain nn.Module so MaseGraph uses
    MaseTracer instead of the unsupported HF symbolic tracer.
    """

    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def forward(self, input_features):
        return self.encoder(input_features).last_hidden_state


model_name = "openai/whisper-tiny"

hf_model = WhisperModel.from_pretrained(model_name, attn_implementation="eager")
processor = WhisperProcessor.from_pretrained(model_name)

model = WhisperEncoderWrapper(hf_model.encoder)
print(model)

/vol/bitbucket/rr422/ADLS/mase/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/vol/bitbucket/rr422/ADLS/mase/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WhisperEncoderWrapper(
  (encoder): WhisperEncoder(
    (conv1): Conv1d(80, 384, kernel_size=(3,), stride=(1,), padding=(1,))
    (conv2): Conv1d(384, 384, kernel_size=(3,), stride=(2,), padding=(1,))
    (embed_positions): Embedding(1500, 384)
    (layers): ModuleList(
      (0-3): 4 x WhisperEncoderLayer(
        (self_attn): WhisperAttention(
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (out_proj): Linear(in_features=384, out_features=384, bias=True)
        )
        (self_attn_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (activation_fn): GELUActivation()
        (fc1): Linear(in_features=384, out_features=1536, bias=True)
        (fc2): Linear(in_features=1536, out_features=384, bias=True)
        (final_layer_norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)

## 2. Generate the FX Graph with MaseGraph

We now trace the Whisper model into a `MaseGraph`. Since Whisper is a HuggingFace `PreTrainedModel`, the `MaseGraph` constructor will automatically use the `HFTracer` to perform symbolic tracing.

We also draw the resulting graph as an SVG for visualization.

In [2]:
import os
import platform

# Add Homebrew path for macOS (Graphviz is typically in system PATH on Linux)
if platform.system() == 'Darwin':  # macOS
    homebrew_bin = '/opt/homebrew/bin'  # ARM64 Mac
    if not os.path.exists(homebrew_bin):
        homebrew_bin = '/usr/local/bin'  # Intel Mac
    if os.path.exists(homebrew_bin):
        os.environ['PATH'] = homebrew_bin + ':' + os.environ.get('PATH', '')

from chop import MaseGraph
from transformers.models.whisper.modeling_whisper import WhisperEncoderLayer

# Provide concrete forward args so the tracer knows the input shapes
cf_args = {
    "input_features": torch.randn(1, 80, 3000),
}

# WhisperEncoderLayer contains attention with control flow that breaks
# symbolic tracing, so we treat it as a leaf (opaque) module.
# The "args" dict maps each forward() parameter to a Mase data role.
# WhisperEncoderLayer.forward(hidden_states, attention_mask, layer_head_mask, output_attentions)
custom_ops = {
    "modules": {
        WhisperEncoderLayer: {
            "args": {
                "hidden_states": "data_in",
                "attention_mask": "config",
                "layer_head_mask": "config",
                "output_attentions": "config",
            },
        },
    },
    "functions": {},
}

mg = MaseGraph(model, cf_args=cf_args, custom_ops=custom_ops)
mg.draw("whisper-tiny-encoder.svg")

/vol/bitbucket/rr422/ADLS/mase/.venv/lib/python3.11/site-packages/torch/fx/_symbolic_trace.py:906: UserWarning: Was not able to add assertion to guarantee correct input input_features to specialized function. It is up to the user to make sure that your inputs match the inputs you specialized the function with.
  warnings.warn(


## 3. Inspect the FX Graph Nodes

Let's print out every node in the traced graph, including the node type (`op`), name, and target. This gives us a bird's-eye view of the Whisper computation graph.

In [3]:
# Print all nodes in the FX graph
print(f"{'Op':<20} {'Name':<55} {'Target':<60}")
print("-" * 135)
for node in mg.fx_graph.nodes:
    print(f"{node.op:<20} {node.name:<55} {str(node.target):<60}")

Op                   Name                                                    Target                                                      
---------------------------------------------------------------------------------------------------------------------------------------
placeholder          input_features_1                                        input_features_1                                            
get_attr             _tensor_constant0                                       _tensor_constant0                                           
call_module          encoder_conv1                                           encoder.conv1                                               
call_function        gelu                                                    <built-in function gelu>                                    
call_module          encoder_conv2                                           encoder.conv2                                               
call_function        gelu_1         

## 4. Summarize node types

Let's count how many nodes of each FX node type (`placeholder`, `call_module`, `call_function`, `call_method`, `get_attr`, `output`) are in the graph.

In [4]:
from collections import Counter

op_counts = Counter(node.op for node in mg.fx_graph.nodes)

print("Node type counts:")
for op, count in sorted(op_counts.items(), key=lambda x: -x[1]):
    print(f"  {op:<20} {count}")

print(f"\nTotal nodes: {sum(op_counts.values())}")

Node type counts:
  call_function        8
  call_module          7
  get_attr             2
  placeholder          1
  call_method          1
  output               1

Total nodes: 20


## 5. Raise the Graph to the Mase IR

To convert the FX graph into the full Mase IR, we run the metadata analysis passes. For the Whisper encoder, we need a dummy `input_features` tensor — a mel spectrogram of shape `(batch, 80, 3000)` (80 mel bins, 30 seconds at 100 frames/s).

In [3]:
import chop.passes as passes

# Create dummy input — the FX tracer renamed the placeholder to "input_features_1"
batch_size = 2
dummy_input = {
    "input_features_1": torch.randn(batch_size, 80, 3000),  # mel spectrogram
}

mg, _ = passes.init_metadata_analysis_pass(mg)
mg, _ = passes.add_common_metadata_analysis_pass(
    mg,
    pass_args={
        "dummy_in": dummy_input,
        "add_value": False,
    },
)

print("Metadata passes completed successfully.")

tensor([[[-1.5728e-01, -1.2112e-01, -6.1803e-03,  ..., -7.8331e-02,
          -1.1965e-01, -1.1391e-01],
         [-3.2815e-04, -9.7156e-04, -3.4460e-06,  ..., -2.1934e-04,
          -3.9238e-04, -3.0050e-04],
         [-1.6514e-01,  4.3315e-01, -1.6987e-01,  ...,  1.0906e+00,
           2.7224e-01,  1.5870e-01],
         ...,
         [-4.0114e-02, -3.1660e-07, -1.6101e-07,  ..., -1.3492e-05,
          -5.0918e-03, -9.1943e-03],
         [-1.6861e-01, -4.1974e-02, -1.6025e-01,  ..., -1.1045e-03,
          -1.3010e-01, -9.0810e-02],
         [-1.0402e-01, -1.2315e-02, -1.6991e-01,  ..., -1.6887e-01,
           7.6992e-01, -6.2981e-04]]], grad_fn=<GeluBackward0>)
Metadata passes completed successfully.


## 6. Inspect Metadata on a Node

Now that we've raised the graph to the Mase IR, each node has metadata attached under `node.meta["mase"]`. Let's inspect the metadata for some encoder and decoder nodes to see shapes, dtypes, and Mase operator types.

In [5]:
# Find and inspect some interesting call_module nodes
inspected = 0
for node in mg.fx_graph.nodes:
    if node.op == "call_module" and inspected < 5:
        mase_meta = node.meta["mase"]
        common = mase_meta["common"]
        print(f"\n--- Node: {node.name} ---")
        print(f"  Target:  {node.target}")
        print(f"  Mase Op: {common.get('mase_op', 'N/A')}")
        if 'results' in common:
            for k, v in common['results'].items():
                if isinstance(v, dict):
                    print(f"  Result '{k}': shape={v.get('shape', 'N/A')}, dtype={v.get('type', 'N/A')}")
        inspected += 1


--- Node: encoder_conv1 ---
  Target:  encoder.conv1
  Mase Op: conv1d
  Result 'data_out_0': shape=[1, 384, 3000], dtype=float

--- Node: encoder_conv2 ---
  Target:  encoder.conv2
  Mase Op: conv1d
  Result 'data_out_0': shape=[1, 384, 1500], dtype=float

--- Node: encoder_layers_0 ---
  Target:  encoder.layers.0
  Mase Op: user_defined_module
  Result 'data_out_0': shape=[1, 1500, 384], dtype=float

--- Node: encoder_layers_1 ---
  Target:  encoder.layers.1
  Mase Op: user_defined_module
  Result 'data_out_0': shape=[1, 1500, 384], dtype=float

--- Node: encoder_layers_2 ---
  Target:  encoder.layers.2
  Mase Op: user_defined_module
  Result 'data_out_0': shape=[1, 1500, 384], dtype=float


## 7. Analysis Pass — Count Encoder Layers

Let's write a simple analysis pass to identify the encoder layer modules and other components (conv layers, layer norms, etc.) in the graph.

In [6]:
from chop.tools import get_logger

logger = get_logger("whisper_analysis")
logger.setLevel("INFO")


def count_components_analysis_pass(mg, pass_args={}):
    """Count encoder layers, conv layers, and other components in the graph."""
    encoder_layers = []
    conv_layers = []
    layer_norms = []
    other_modules = []

    for node in mg.fx_graph.nodes:
        if node.op == "call_module":
            target = node.target
            if "layers" in target and target.count(".") == 1:
                # Top-level encoder layer (e.g., "encoder.layers.0")
                encoder_layers.append(target)
            elif "conv" in target:
                conv_layers.append(target)
            elif "layer_norm" in target:
                layer_norms.append(target)
            else:
                other_modules.append(target)

    results = {
        "encoder_layers": encoder_layers,
        "conv_layers": conv_layers,
        "layer_norms": layer_norms,
        "other_modules": other_modules,
    }

    logger.info(f"Encoder transformer layers: {len(encoder_layers)}")
    logger.info(f"Conv layers: {len(conv_layers)}")
    logger.info(f"Layer norm layers: {len(layer_norms)}")
    logger.info(f"Other modules: {len(other_modules)}")

    return mg, results


mg, component_results = count_components_analysis_pass(mg)

INFO     Encoder transformer layers: 0
INFO     Conv layers: 2
INFO     Layer norm layers: 1
INFO     Other modules: 4


## 8. Analysis Pass — Count Dropout Layers

In the Whisper encoder, dropout at the top level is applied via `nn.functional.dropout` (a `call_function` node), while dropout inside the encoder layers is hidden since those are leaf modules. Let's count both `call_module` and `call_function` dropout nodes.

In [8]:
def count_dropout_analysis_pass(mg, pass_args={}):
    dropout_count = 0
    for node in mg.fx_graph.nodes:
        # Check for both call_module and call_function dropout
        if node.op == "call_module" and "dropout" in node.target:
            logger.info(f"Found dropout module: {node.target}")
            dropout_count += 1
        elif node.op == "call_function" and "dropout" in str(node.target):
            logger.info(f"Found dropout function: {node.name} -> {node.target}")
            dropout_count += 1
    return mg, {"dropout_count": dropout_count}


mg, pass_out = count_dropout_analysis_pass(mg)
logger.info(f"Total dropout nodes: {pass_out['dropout_count']}")

INFO     Found dropout function: dropout -> <function dropout at 0x7acc0a990f40>
INFO     Total dropout nodes: 1


## 9. Transform Pass — Remove Dropout

As in Tutorial 1, dropout has no effect at inference time. Here we remove both `call_module` and `call_function` dropout nodes from the graph.

In [9]:
import torch.fx as fx


def remove_dropout_transform_pass(mg, pass_args={}):
    for node in mg.fx_graph.nodes:
        is_dropout_module = node.op == "call_module" and "dropout" in node.target
        is_dropout_function = node.op == "call_function" and "dropout" in str(node.target)

        if is_dropout_module or is_dropout_function:
            logger.info(f"Removing dropout node: {node.name}")
            # The first arg is the input tensor — replace all uses with it
            parent_node = node.args[0]
            node.replace_all_uses_with(parent_node)
            mg.fx_graph.erase_node(node)
    return mg, {}


mg, _ = remove_dropout_transform_pass(mg)
mg, pass_out = count_dropout_analysis_pass(mg)

assert pass_out["dropout_count"] == 0
logger.info("All dropout nodes removed successfully.")

INFO     Removing dropout node: dropout
INFO     All dropout nodes removed successfully.


## 10. Export the MaseGraph

Finally, export the transformed graph so it can be reloaded later.

In [10]:
from pathlib import Path

export_path = f"{Path.home()}/whisper_mase_graph"
mg.export(export_path)
print(f"MaseGraph exported to {export_path}")

INFO     Exporting MaseGraph to /homes/rr422/whisper_mase_graph.pt, /homes/rr422/whisper_mase_graph.mz
INFO     Exporting GraphModule to /homes/rr422/whisper_mase_graph.pt
INFO     Saving full model format
INFO     Exporting MaseMetadata to /homes/rr422/whisper_mase_graph.mz


MaseGraph exported to /homes/rr422/whisper_mase_graph


In [11]:
# Verify we can reload it
new_mg = MaseGraph.from_checkpoint(export_path)
print(f"Reloaded graph has {len(list(new_mg.fx_graph.nodes))} nodes.")

Reloaded graph has 19 nodes.
